# YAGO3-10 Full Experiment (GPU Required)

**Run on Google Colab with GPU runtime.**

This notebook runs:
1. Coverage-only baseline
2. VanillaGPKGE
3. CAGP

Expected time: ~30 minutes on T4 GPU

In [1]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q scikit-learn

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score
import json
import os
import random
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Configuration
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'kl_weight': 0.01,
    'seeds': [42, 123, 456],
}
print(f"Config: {CONFIG}")

Config: {'epochs': 50, 'embedding_dim': 100, 'batch_size': 2048, 'lr': 0.001, 'kl_weight': 0.01, 'seeds': [42, 123, 456]}


In [5]:
# Download YAGO3-10
def download_yago():
    os.makedirs('data', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/YAGO3-10"

    for split in ['train', 'test', 'valid']:
        path = f'data/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("Download complete!")

download_yago()

Download complete!


In [6]:
def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples('data/train.txt')
test = load_triples('data/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

print(f"YAGO3-10 Statistics:")
print(f"  Train: {len(train):,}")
print(f"  Test: {len(test):,}")
print(f"  Entities: {len(entities):,}")
print(f"  Relations: {len(relations)}")

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}

YAGO3-10 Statistics:
  Train: 1,079,040
  Test: 5,000
  Entities: 123,161
  Relations: 37


In [7]:
class CoverageOnlyDetector:
    """Pure coverage-based uncertainty (no learning)."""
    def __init__(self, num_entities, num_relations):
        self.coverage = torch.zeros(num_entities, num_relations)

    def fit(self, triples, entity_to_idx, relation_to_idx):
        for h, r, t in triples:
            self.coverage[entity_to_idx[h], relation_to_idx[r]] = 1.0
            self.coverage[entity_to_idx[t], relation_to_idx[r]] = 1.0
        return self

    def get_uncertainty(self, heads, relations, tails):
        h_seen = self.coverage[heads.cpu(), relations.cpu()]
        t_seen = self.coverage[tails.cpu(), relations.cpu()]
        return (2.0 - h_seen - t_seen).to(heads.device)

In [8]:
class VanillaGPKGE(nn.Module):
    """GP-KGE with entity-level variance."""
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        return (h_var + t_var) / 2

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities

In [9]:
class CAGP(nn.Module):
    """Coverage-Augmented GP-KGE."""
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations

        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))  # sigmoid(0) = 0.5

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        # GP variance (semantic)
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2

        # Coverage (structural)
        h_seen = self.coverage[heads, relations]
        t_seen = self.coverage[tails, relations]
        cov_unc = 2.0 - h_seen - t_seen

        # Normalize GP variance to same scale
        gp_var_norm = gp_var / (gp_var.mean() + 1e-8) * (cov_unc.mean() + 1e-8)

        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_var_norm + (1 - alpha) * cov_unc

    def precompute_coverage(self, triples, entity_to_idx, relation_to_idx):
        for h, r, t in triples:
            self.coverage[entity_to_idx[h], relation_to_idx[r]] = 1.0
            self.coverage[entity_to_idx[t], relation_to_idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities

    def get_alpha(self):
        return torch.sigmoid(self.alpha_logit).item()

In [10]:
def train_model(model, triples, entity_to_idx, relation_to_idx, epochs, is_gp=False):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    if hasattr(model, 'precompute_coverage'):
        model.precompute_coverage(triples, entity_to_idx, relation_to_idx)

    heads = torch.tensor([entity_to_idx[h] for h, r, t in triples])
    relations = torch.tensor([relation_to_idx[r] for h, r, t in triples])
    tails = torch.tensor([entity_to_idx[t] for h, r, t in triples])

    num_entities = len(entity_to_idx)
    dataset = TensorDataset(heads, relations, tails)
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos_scores = model(batch_h, batch_r, batch_t, use_sampling=is_gp)
            neg_t = torch.randint(0, num_entities, batch_t.shape, device=device)
            neg_scores = model(batch_h, batch_r, neg_t, use_sampling=is_gp)

            loss = criterion(pos_scores, torch.ones_like(pos_scores)) + \
                   criterion(neg_scores, torch.zeros_like(neg_scores))

            if is_gp and hasattr(model, 'kl_loss'):
                loss += CONFIG['kl_weight'] * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return model

In [11]:
def evaluate_auroc(model, test_triples, entity_to_idx, relation_to_idx, is_coverage_only=False):
    if not is_coverage_only:
        model.eval()

    heads = torch.tensor([entity_to_idx.get(h, 0) for h, r, t in test_triples]).to(device)
    relations = torch.tensor([relation_to_idx.get(r, 0) for h, r, t in test_triples]).to(device)
    tails = torch.tensor([entity_to_idx.get(t, 0) for h, r, t in test_triples]).to(device)

    with torch.no_grad():
        id_unc = model.get_uncertainty(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(entity_to_idx), tails.shape, device=device)
        ood_unc = model.get_uncertainty(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])
    return roc_auc_score(labels, scores)

In [12]:
# Run experiments
results = {
    'CoverageOnly': [],
    'VanillaGPKGE': [],
    'CAGP': []
}
alphas = []

for seed in CONFIG['seeds']:
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print('='*50)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # Coverage Only
    print("\n1. Coverage Only...")
    cov_detector = CoverageOnlyDetector(len(ent2idx), len(rel2idx))
    cov_detector.fit(train, ent2idx, rel2idx)
    auroc_cov = evaluate_auroc(cov_detector, test, ent2idx, rel2idx, is_coverage_only=True)
    results['CoverageOnly'].append(auroc_cov)
    print(f"   AUROC: {auroc_cov:.4f}")

    # Vanilla GP-KGE
    print("\n2. VanillaGPKGE...")
    gp = VanillaGPKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    gp = train_model(gp, train, ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_gp = evaluate_auroc(gp, test, ent2idx, rel2idx)
    results['VanillaGPKGE'].append(auroc_gp)
    print(f"   AUROC: {auroc_gp:.4f}")

    # CAGP
    print("\n3. CAGP...")
    cagp = CAGP(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    cagp = train_model(cagp, train, ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_cagp = evaluate_auroc(cagp, test, ent2idx, rel2idx)
    results['CAGP'].append(auroc_cagp)
    alphas.append(cagp.get_alpha())
    print(f"   AUROC: {auroc_cagp:.4f} (alpha={cagp.get_alpha():.3f})")


Seed 42

1. Coverage Only...
   AUROC: 0.7630

2. VanillaGPKGE...
  Epoch 10/50, Loss: 1.4488
  Epoch 20/50, Loss: 1.4001
  Epoch 30/50, Loss: 1.3918
  Epoch 40/50, Loss: 1.3294
  Epoch 50/50, Loss: 0.9594
   AUROC: 0.8270

3. CAGP...
  Epoch 10/50, Loss: 1.4480
  Epoch 20/50, Loss: 1.3999
  Epoch 30/50, Loss: 1.3636
  Epoch 40/50, Loss: 0.9994
  Epoch 50/50, Loss: 0.7426
   AUROC: 0.9425 (alpha=0.500)

Seed 123

1. Coverage Only...
   AUROC: 0.7573

2. VanillaGPKGE...
  Epoch 10/50, Loss: 1.4486
  Epoch 20/50, Loss: 1.4002
  Epoch 30/50, Loss: 1.3886
  Epoch 40/50, Loss: 1.3785
  Epoch 50/50, Loss: 1.2308
   AUROC: 0.8274

3. CAGP...
  Epoch 10/50, Loss: 1.4491
  Epoch 20/50, Loss: 1.4000
  Epoch 30/50, Loss: 1.3875
  Epoch 40/50, Loss: 1.2406
  Epoch 50/50, Loss: 0.9042
   AUROC: 0.9422 (alpha=0.500)

Seed 456

1. Coverage Only...
   AUROC: 0.7598

2. VanillaGPKGE...
  Epoch 10/50, Loss: 1.4483
  Epoch 20/50, Loss: 1.4000
  Epoch 30/50, Loss: 1.2828
  Epoch 40/50, Loss: 0.9040
  Epo

In [13]:
# Summary
print("\n" + "="*70)
print("YAGO3-10 FINAL RESULTS (37 relations)")
print("="*70)
print(f"{'Method':<20} {'AUROC':<15} {'vs Best Single'}")
print("-"*50)

best_single = max(np.mean(results['CoverageOnly']), np.mean(results['VanillaGPKGE']))

for method in ['CoverageOnly', 'VanillaGPKGE', 'CAGP']:
    mean = np.mean(results[method])
    std = np.std(results[method])
    delta = mean - best_single
    print(f"{method:<20} {mean:.4f} ± {std:.3f}   {delta:+.4f}")

print("-"*50)
print(f"Learned alpha: {np.mean(alphas):.3f} ± {np.std(alphas):.3f}")

synergy = np.mean(results['CAGP']) - best_single
print(f"\nSynergy: {synergy:+.4f} ({synergy/best_single*100:+.1f}%)")

if synergy > 0.05:
    print("\n** SYNERGY CONFIRMED: CAGP >> best single component **")
elif synergy > 0.02:
    print("\n** MODERATE SYNERGY: CAGP > best single component **")
else:
    print("\n** WEAK/NO SYNERGY **")


YAGO3-10 FINAL RESULTS (37 relations)
Method               AUROC           vs Best Single
--------------------------------------------------
CoverageOnly         0.7600 ± 0.002   -0.0642
VanillaGPKGE         0.8242 ± 0.004   +0.0000
CAGP                 0.9424 ± 0.000   +0.1182
--------------------------------------------------
Learned alpha: 0.500 ± 0.000

Synergy: +0.1182 (+14.3%)

** SYNERGY CONFIRMED: CAGP >> best single component **


In [14]:
# Save results
output = {
    'dataset': 'YAGO3-10',
    'num_entities': len(ent2idx),
    'num_relations': len(rel2idx),
    'config': CONFIG,
    'results': {
        m: {'mean': float(np.mean(results[m])), 'std': float(np.std(results[m]))}
        for m in results
    },
    'learned_alpha': {'mean': float(np.mean(alphas)), 'std': float(np.std(alphas))},
    'synergy': float(synergy)
}

with open('yago_full_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nResults saved to yago_full_results.json")
print(json.dumps(output, indent=2))


Results saved to yago_full_results.json
{
  "dataset": "YAGO3-10",
  "num_entities": 123161,
  "num_relations": 37,
  "config": {
    "epochs": 50,
    "embedding_dim": 100,
    "batch_size": 2048,
    "lr": 0.001,
    "kl_weight": 0.01,
    "seeds": [
      42,
      123,
      456
    ]
  },
  "results": {
    "CoverageOnly": {
      "mean": 0.7600461866666666,
      "std": 0.002362534625147827
    },
    "VanillaGPKGE": {
      "mean": 0.8241968066666666,
      "std": 0.00422834723371784
    },
    "CAGP": {
      "mean": 0.9423832666666666,
      "std": 0.00011530363085736189
    }
  },
  "learned_alpha": {
    "mean": 0.5,
    "std": 0.0
  },
  "synergy": 0.11818646
}
